# Create American Heart Association Awards (GRANT PATTERN, partner Excel export)

The **American Heart Association** (AHA, F4320306230, US) is the largest
non-government funder of cardiovascular and cerebrovascular research in the US
(~$2B of awards 2015-2026 in this export). As part of the OpenAlex/AHA
collaboration — AHA is moving compliance monitoring + funding intelligence off
Dimensions onto open data — AHA sends us a **periodic Excel export of their own
award metadata straight from their grants-management system** ("Report
Builder" / RB pull).

The source script `scripts/local/aha_to_s3.py` reads that export's **RB tab**
(AHA's authoritative pull) and normalises it to the award schema. It does NOT
ingest the workbook's second "Dimensions pull" tab: that tab carries an
explicit Digital Science copyright + redistribution restriction and is an
all-but-exact subset of the RB pull (~10,201 of 10,203 grant numbers already in
RB), so ingesting it would mean redistributing copyrighted data for a net gain
of ~2 records. It is retained only as a private QA cross-check.

**Awarding body:** American Heart Association — F4320306230 (US), ROR
`https://ror.org/013kjyp64`, DOI `10.13039/100000968`. Path A (F4320*
Crossref-registered funder, present in `openalex.common.funder`).

**Multi-PI collaborative awards:** AHA's collaborative mechanisms
(Collaborative Sciences Award, Strategically Focused Research Networks, ...)
list one award id across multiple PI rows — one per co-PI at a different
institution, each with a portion of the total. The script **aggregates each
award id into one award row**: `amount` = sum of the per-PI portions, and all
PIs are carried in `investigators[]` (first-listed = `lead_investigator`). This
is why the parquet is one-row-per-award with an `investigators_json` column.

**Schema choices / known limitations:**
- One row per award; `funder_award_id` = AHA's own award number
  (e.g. `15BGIA22410018`, `26HTRN1673063`).
- `currency = 'USD'` hardcoded (US funder); ~99.8% of awards publish an amount.
  ~21 awards publish a $0 total (declined/withdrawn) -> `amount` NULL.
- `description` = scientific ("Technical") abstract, falling back to the lay
  ("General") abstract; ~99% populated.
- **EXACT `start_date` / `end_date`** are published (100%) -> both populated,
  and `start_year`/`end_year` derived from them.
- `funder_scheme` = the "Funding Mechanism" string; `funding_type` derived
  (Predoctoral/Undergraduate/Supplement/SURE -> training;
  Fellowship/Career Development/Scientist Development -> fellowship;
  else research).
- PI names come as clean First/Last columns (degrees are separate) -> no
  honorific stripping; **no ORCIDs** published -> `orcid` NULL. PI degrees and
  academic rank have no home in the award schema and are dropped.
- `affiliation.country` = 'US' when the institution has a US state anywhere in
  the export; genuinely international awardees (e.g. International Visiting
  Professorship hosts) stay NULL.

**Prerequisites:** run `scripts/local/aha_to_s3.py --input <AHA export>.xlsx`
first to build + upload the parquet to S3.

**Data source:** American Heart Association Report Builder export (partner-supplied)
**S3 location:** `s3a://openalex-ingest/awards/aha/aha_projects.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aha_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aha/aha_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.aha_raw;

## Step 1.5: Inspect raw + money/currency scan

Per runbook §1.5, scan for money-flavored columns even though the script
already parsed amounts. Known source columns (all STRING per §1.2.5):
`funder_award_id`, `title`, `description`, `amount`, `currency`,
`funder_scheme`, `program_name`, `funding_type`, `cycle`, `start_date`,
`end_date`, `start_year`, `end_year`, `n_investigators`, `investigators_json`,
`landing_page_url`.

In [ ]:
%sql
DESCRIBE openalex.awards.aha_raw;

In [ ]:
%sql
SELECT funder_award_id, title, amount, currency, funder_scheme, funding_type,
       start_date, end_date, n_investigators, SUBSTRING(investigators_json, 1, 120) AS investigators
FROM openalex.awards.aha_raw
LIMIT 5;

In [ ]:
%sql
-- Money-shape scan per §1.5: amount already parsed; confirm USD range.
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total
FROM openalex.awards.aha_raw;

In [ ]:
%sql
-- Currency present (should be only 'USD' or NULL).
SELECT currency, COUNT(*) AS rows FROM openalex.awards.aha_raw GROUP BY currency;

## Step 1.6: Fail-fast — verify AHA funder row exists (Path A, F4320*)

The Step 2 transform `CROSS JOIN`s against `openalex.common.funder`. AHA is a
Crossref-registered F4320* funder, so this **must return exactly 1 row** with
ror_id `https://ror.org/013kjyp64` and doi `10.13039/100000968`. If 0, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306230;  -- American Heart Association

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aha_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306230  -- American Heart Association
),
parsed AS (
    SELECT
        a.*,
        from_json(
            a.investigators_json,
            'array<struct<given_name:string,family_name:string,orcid:string,role_start:date,affiliation:struct<name:string,country:string,ids:array<struct<id:string,type:string,asserted_by:string>>>>>'
        ) AS investigators_arr
    FROM openalex.awards.aha_raw a
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(p.funder_award_id)
    ))) % 9000000000 AS id,
    p.title AS display_name,
    p.description,                                 -- Technical abstract (General fallback)
    f.funder_id,
    p.funder_award_id,
    TRY_CAST(p.amount AS DOUBLE) AS amount,
    p.currency,                                    -- 'USD' from script, NULL if no amount
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    p.funding_type,                                -- research / fellowship / training
    p.funder_scheme,                               -- AHA funding mechanism
    'aha_report_builder' AS provenance,
    TRY_CAST(p.start_date AS DATE) AS start_date,  -- exact award dates (published)
    TRY_CAST(p.end_date AS DATE) AS end_date,
    TRY_CAST(p.start_year AS INT) AS start_year,
    TRY_CAST(p.end_year AS INT) AS end_year,
    -- first-listed PI = lead / primary applicant
    try_element_at(p.investigators_arr, 1) AS lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) AS co_lead_investigator,
    p.investigators_arr AS investigators,          -- all PIs (multi-PI collaboratives)
    p.landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(p.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM parsed p
CROSS JOIN funder_resolved f
WHERE p.funder_award_id IS NOT NULL
  AND p.title IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 397

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aha_report_builder' AND priority = 397;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    397 AS priority  -- AHA priority (next odd slot above BHF 395; HIGHER wins per oxjob #500)
FROM openalex.awards.aha_awards;

## Step 6: Verification

Full §6.1–6.8 verification. AHA amount-coverage is NOT waived — expect ~99.8%
`pct_amount`, single currency `'USD'`, exact start/end dates, and a real
long-tail of PI family names (never an institution word).

In [ ]:
%sql
SELECT COUNT(*) AS total_aha_award_rows FROM openalex.awards.aha_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.aha_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(lead_investigator) AS has_lead_pi,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead_pi
FROM openalex.awards.aha_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast check (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount, AVG(amount) AS avg_amount,
    ROUND(SUM(amount) / 1e9, 2) AS total_usd_billion
FROM openalex.awards.aha_awards;

In [ ]:
%sql
-- §6.4 sample inspection (incl. lead PI + investigator count)
SELECT id, SUBSTRING(display_name, 1, 55) AS title, funder_scheme, funding_type,
       amount, currency, start_date, end_date,
       lead_investigator.given_name AS lead_given,
       lead_investigator.family_name AS lead_family,
       lead_investigator.affiliation.name AS lead_institution,
       size(investigators) AS n_pi
FROM openalex.awards.aha_awards
ORDER BY start_date DESC NULLS LAST, amount DESC NULLS LAST
LIMIT 12;

In [ ]:
%sql
-- §6.4a PI frequency check — top family names must be a real long-tail,
-- never an institution word.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.aha_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Multi-PI collaborative awards — confirm investigators[] is populated (>1).
SELECT size(investigators) AS n_pi, COUNT(*) AS awards
FROM openalex.awards.aha_awards
GROUP BY size(investigators) ORDER BY n_pi;

In [ ]:
%sql
-- Sample a multi-PI collaborative award end-to-end.
SELECT funder_award_id, display_name, amount,
       TRANSFORM(investigators, x -> CONCAT(x.given_name, ' ', x.family_name,
                 ' (', x.affiliation.name, ')')) AS pis
FROM openalex.awards.aha_awards
WHERE size(investigators) > 1
ORDER BY size(investigators) DESC, amount DESC
LIMIT 5;

In [ ]:
%sql
-- Scheme distribution + funding_type split
SELECT funder_scheme, funding_type, COUNT(*) AS rows, ROUND(SUM(amount)/1e6, 1) AS total_usd_m
FROM openalex.awards.aha_awards
GROUP BY funder_scheme, funding_type ORDER BY rows DESC LIMIT 25;

In [ ]:
%sql
-- §6.6 year distribution — expect ~2015–2026, ~500–1,200 awards/yr
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.aha_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
-- §6.5 funder consistency — should be exactly American Heart Association
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.aha_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- §6.8 confirm rows reached the shared raw table at the right priority
SELECT provenance, priority, COUNT(*) AS n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aha_report_builder'
GROUP BY provenance, priority;